In [1]:
import json
import random
from collections import defaultdict

In [2]:

# ── Configuration ─────────────────────────────────────────────────────────────
# Raw upstream candidate pool (full schema: profile/career_history/skills/...).
# This is the un-sampled pool the train/val/test splits below are drawn FROM --
# point it at your actual ~12k-candidate source file.
INPUT_FILE = "../data/final_candidates_12yoe.jsonl"


In [4]:
TARGET_SAMPLE_SIZE = 800
TRAIN_SIZE = 560
VAL_SIZE = 120
TEST_SIZE = 120

assert TRAIN_SIZE + VAL_SIZE + TEST_SIZE == TARGET_SAMPLE_SIZE

TRAIN_FILE = "../data/train_set.jsonl"
VAL_FILE = "../data/validation_set.jsonl"
TEST_FILE = "../data/test_set.jsonl"

# The JD targets 5-9 years, so we build buckets around that core requirement.
BUCKET_NAMES = ["Junior (<5)", "Target (5-9)", "Senior (10+)"]


def get_yoe_bucket(yoe):
    """Categorize candidates into experience buckets."""
    if yoe < 5:
        return BUCKET_NAMES[0]
    elif 5 <= yoe <= 9:
        return BUCKET_NAMES[1]
    else:
        return BUCKET_NAMES[2]


def save_jsonl(filename, candidates):
    """Save candidates to a JSONL file."""
    with open(filename, "w", encoding="utf-8") as f:
        for candidate in candidates:
            f.write(json.dumps(candidate) + "\n")


def print_distribution(split_name, candidates):
    """Print YOE distribution for a dataset split."""
    counts = {bucket: 0 for bucket in BUCKET_NAMES}

    for c in candidates:
        yoe = float(c.get("profile", {}).get("years_of_experience") or 0)
        counts[get_yoe_bucket(yoe)] += 1

    print(f"\n── {split_name} ({len(candidates)} candidates) ──")
    for bucket in BUCKET_NAMES:
        print(f"  {bucket:<15}: {counts[bucket]}")


def main():
    random.seed(42)

    # --------------------------------------------------
    # 1. Read candidates
    # --------------------------------------------------
    buckets = defaultdict(list)
    total_loaded = 0

    print(f"[*] Reading candidates from {INPUT_FILE}...")

    try:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                candidate = json.loads(line)

                yoe = float(
                    candidate.get("profile", {}).get("years_of_experience") or 0
                )

                bucket = get_yoe_bucket(yoe)
                buckets[bucket].append(candidate)
                total_loaded += 1

    except FileNotFoundError:
        print(f"[!] Error: {INPUT_FILE} not found.")
        return

    print(f"\n[*] Loaded {total_loaded:,} candidates.")
    print("\n── Initial Bucket Distribution ──")
    for bucket in BUCKET_NAMES:
        print(f"  {bucket:<15}: {len(buckets[bucket])}")

    # --------------------------------------------------
    # 2. Sample equally from each bucket
    # --------------------------------------------------
    target_per_bucket = TARGET_SAMPLE_SIZE // len(BUCKET_NAMES)

    sampled_candidates = []

    for bucket in BUCKET_NAMES:
        candidates = buckets[bucket]
        random.shuffle(candidates)

        take = min(len(candidates), target_per_bucket)
        sampled_candidates.extend(candidates[:take])

    # --------------------------------------------------
    # 3. Fill any remaining shortfall
    # --------------------------------------------------
    shortfall = TARGET_SAMPLE_SIZE - len(sampled_candidates)

    if shortfall > 0:
        print(f"\n[*] Resolving shortfall of {shortfall} candidates...")

        sampled_ids = {
            candidate["candidate_id"]
            for candidate in sampled_candidates
        }

        remaining_pool = []

        for bucket in BUCKET_NAMES:
            for candidate in buckets[bucket]:
                if candidate["candidate_id"] not in sampled_ids:
                    remaining_pool.append(candidate)

        random.shuffle(remaining_pool)
        sampled_candidates.extend(remaining_pool[:shortfall])

    # Safety check
    sampled_candidates = sampled_candidates[:TARGET_SAMPLE_SIZE]

    print(f"\n[*] Final sampled candidates: {len(sampled_candidates)}")

    # --------------------------------------------------
    # 4. Final shuffle
    # --------------------------------------------------
    random.shuffle(sampled_candidates)

    # --------------------------------------------------
    # 5. Train / Validation / Test split
    # --------------------------------------------------
    train_candidates = sampled_candidates[:TRAIN_SIZE]

    val_candidates = sampled_candidates[
        TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE
    ]

    test_candidates = sampled_candidates[
        TRAIN_SIZE + VAL_SIZE:
        TRAIN_SIZE + VAL_SIZE + TEST_SIZE
    ]

    # --------------------------------------------------
    # 6. Save datasets
    # --------------------------------------------------
    save_jsonl(TRAIN_FILE, train_candidates)
    save_jsonl(VAL_FILE, val_candidates)
    save_jsonl(TEST_FILE, test_candidates)

    # --------------------------------------------------
    # 7. Print distributions
    # --------------------------------------------------
    print_distribution("Train", train_candidates)
    print_distribution("Validation", val_candidates)
    print_distribution("Test", test_candidates)

    print("\n==============================")
    print("Dataset successfully created!")
    print("==============================")
    print(f"Train      : {len(train_candidates)} -> {TRAIN_FILE}")
    print(f"Validation : {len(val_candidates)} -> {VAL_FILE}")
    print(f"Test       : {len(test_candidates)} -> {TEST_FILE}")


In [11]:
main()


[*] Reading candidates from ../data/final_candidates_12yoe.jsonl...

[*] Loaded 4,896 candidates.

── Initial Bucket Distribution ──
  Junior (<5)    : 1316
  Target (5-9)   : 2875
  Senior (10+)   : 705

[*] Resolving shortfall of 2 candidates...

[*] Final sampled candidates: 800

── Train (560 candidates) ──
  Junior (<5)    : 188
  Target (5-9)   : 183
  Senior (10+)   : 189

── Validation (120 candidates) ──
  Junior (<5)    : 39
  Target (5-9)   : 45
  Senior (10+)   : 36

── Test (120 candidates) ──
  Junior (<5)    : 40
  Target (5-9)   : 39
  Senior (10+)   : 41

Dataset successfully created!
Train      : 560 -> ../data/train_set.jsonl
Validation : 120 -> ../data/validation_set.jsonl
Test       : 120 -> ../data/test_set.jsonl


In [3]:
import os


DATA_DIR = "../data"

GOLDEN_SET_PATH = os.path.join(DATA_DIR, "golden_set.jsonl")
CANDIDATE_POOL_FILES = [
    os.path.join(DATA_DIR, "train_set.jsonl"),
    os.path.join(DATA_DIR, "validation_set.jsonl"),
    os.path.join(DATA_DIR, "test_set.jsonl"),
]

def ensure_golden_set_file(
    golden_path=GOLDEN_SET_PATH,
    split_files=(
        os.path.join(DATA_DIR, "golden_train.jsonl"),
        os.path.join(DATA_DIR, "golden_validation.jsonl"),
        os.path.join(DATA_DIR, "golden_test.jsonl"),
    ),
):
    """If a combined golden_set.jsonl isn't present, build it by concatenating
    the per-split golden label files (golden_train.jsonl / golden_validation.jsonl /
    golden_test.jsonl), which is what the labeling pipeline actually writes."""
    if os.path.exists(golden_path):
        return golden_path
    found = [p for p in split_files if os.path.exists(p)]
    if not found:
        print(f"[!] Warning: neither {golden_path} nor split files {list(split_files)} were found.")
        return golden_path
    print(f"[*] {golden_path} not found -- building it from: {found}")
    with open(golden_path, "w", encoding="utf-8") as out:
        for p in found:
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        out.write(line if line.endswith("\n") else line + "\n")
    return golden_path


def load_candidate_pool(paths):
    """Load one or more full-candidate JSONL files into a dict keyed by candidate_id."""
    pool = {}
    for path in paths:
        if not os.path.exists(path):
            print(f"[!] Warning: candidate pool file not found, skipping: {path}")
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                pool[rec["candidate_id"]] = rec
    return pool


def normalize_label_record(label_rec):
    """Golden label files come in two shapes depending on which stage of the
    pipeline wrote them:
      (a) {"candidate_id": ..., "parsed": {Tech_Fit, Context_Fit, ...}}  -- the
          shape create_testset.py's batch pipeline writes to golden_set.jsonl.
      (b) {"Candidate_ID": ..., "Tech_Fit": ..., "Context_Fit": ...}     -- a
          flat per-split shape (golden_train.jsonl / golden_validation.jsonl /
          golden_test.jsonl as delivered).
    Normalize both into (candidate_id, parsed_dict)."""
    if "parsed" in label_rec:
        cid = label_rec.get("candidate_id") or label_rec.get("Candidate_ID")
        return cid, label_rec.get("parsed")
    if "Tech_Fit" in label_rec or "Candidate_ID" in label_rec:
        cid = label_rec.get("Candidate_ID") or label_rec.get("candidate_id")
        return cid, label_rec
    return None, None


def load_golden_set(golden_path=GOLDEN_SET_PATH, pool_paths=CANDIDATE_POOL_FILES):
    """Return a list of full candidate records (profile/career_history/skills/
    redrob_signals/...) each enriched with the golden "parsed" labels, keyed
    and merged by candidate_id. Handles both golden-label record shapes (see
    normalize_label_record)."""
    golden_path = ensure_golden_set_file(golden_path)
    pool = load_candidate_pool(pool_paths)

    merged = []
    skipped = 0
    with open(golden_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            label_rec = json.loads(line)
            cid, parsed = normalize_label_record(label_rec)
            if not cid or not parsed:
                skipped += 1
                continue
            full = pool.get(cid)
            if full is None:
                skipped += 1
                continue
            record = dict(full)  # profile, career_history, skills, redrob_signals, ...
            record["candidate_id"] = cid
            record["parsed"] = parsed
            merged.append(record)

    if skipped:
        print(f"[!] Skipped {skipped} golden-set record(s) with no matching candidate in the pool or no label.")
    return merged


Tech Fit Approches

- Embeddings + Classical ML

In [4]:
import json
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix
from sentence_transformers import SentenceTransformer

def extract_candidate_text(candidate):
    """Combine text fields to feed into the embedding model."""
    profile = candidate.get("profile", {})
    text_parts = [
        profile.get("headline", ""),
        profile.get("summary", "")
    ]
    
    for role in candidate.get("career_history", []):
        text_parts.append(role.get("title", ""))
        text_parts.append(role.get("description", ""))
        
    for skill in candidate.get("skills", []):
        text_parts.append(skill.get("name", ""))
        
    return " ".join(filter(None, text_parts))

def main():
    # 1. Load Golden Set Data (merged with full candidate records -- see the
    #    shared loader cell above for why this merge is necessary)
    texts = []
    labels = []
    candidate_ids = []

    print(f"[*] Loading dataset from {GOLDEN_SET_PATH} (merged with {CANDIDATE_POOL_FILES})...")
    for record in load_golden_set():
        parsed = record.get("parsed", {})
        tech_fit = parsed.get("Tech_Fit")

        # Fallback if your JSON uses a different key name for the label
        if tech_fit is None:
            continue

        candidate_ids.append(record["candidate_id"])
        texts.append(extract_candidate_text(record))
        labels.append(int(tech_fit))

    if not texts:
        print("[!] No valid labels found. Check your JSON keys.")
        return

    print(f"[*] Loaded {len(texts)} labeled candidates.")
    
    # 2. Generate Dense Embeddings
    print("[*] Generating embeddings using all-MiniLM-L6-v2 (this takes a few seconds)...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    X = model.encode(texts, show_progress_bar=True)
    y = np.array(labels)

    # 3. Train / Test Split (80% Train, 20% Test)
    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, candidate_ids, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"\n[*] Training XGBoost Classifier on {len(X_train)} samples...")
    
    # Configure XGBoost for Multi-class classification (0, 1, 2, 3, 4)
    clf = xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=5,
        eval_metric='mlogloss',
        max_depth=3,          # Keep trees shallow to prevent overfitting on small data
        learning_rate=0.1,
        n_estimators=100,
        random_state=42
    )
    
    clf.fit(X_train, y_train)

    # 4. Evaluate the Model
    print("\n[*] Evaluating on Test Set...")
    y_pred = clf.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    print("=" * 40)
    print(" ── APPROACH 2: EVALUATION METRICS ── ")
    print("=" * 40)
    print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")
    print(f"Mean Absolute Error:  {mae:.4f}")
    print("=" * 40)
    
    print("\nConfusion Matrix (Rows: True Tier, Cols: Predicted Tier):")
    print(confusion_matrix(y_test, y_pred))
    
    # Show exactly where the model failed
    print("\n[*] Reviewing Errors on Test Set:")
    error_count = 0
    for cid, true_y, pred_y in zip(ids_test, y_test, y_pred):
        if true_y != pred_y:
            error_count += 1
            print(f"  Candidate: {cid} | True: {true_y} | Predicted: {pred_y}")
            
    if error_count == 0:
        print("  Perfect prediction on test set!")

if __name__ == "__main__":
    main()

[*] Loading dataset from ../data\golden_set.jsonl (merged with ['../data\\train_set.jsonl', '../data\\validation_set.jsonl', '../data\\test_set.jsonl'])...
[*] Loaded 800 labeled candidates.
[*] Generating embeddings using all-MiniLM-L6-v2 (this takes a few seconds)...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]


[*] Training XGBoost Classifier on 640 samples...

[*] Evaluating on Test Set...
 ── APPROACH 2: EVALUATION METRICS ── 
Exact Match Accuracy: 98.12%
Mean Absolute Error:  0.0375

Confusion Matrix (Rows: True Tier, Cols: Predicted Tier):
[[ 34   0   0   0   0]
 [  0   2   1   0   0]
 [  0   0 115   0   0]
 [  0   0   0   1   0]
 [  0   1   1   0   5]]

[*] Reviewing Errors on Test Set:
  Candidate: CAND_0025882 | True: 4 | Predicted: 1
  Candidate: CAND_0048265 | True: 1 | Predicted: 2
  Candidate: CAND_0097176 | True: 4 | Predicted: 2


- Zero-Shot NLI (Natural Language Inference)

In [7]:
import json
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix

def extract_candidate_text(candidate):
    """
    Combines text fields. NLI models usually have a 512-token limit.
    We prioritize the most recent job description and skills.
    """
    text_parts = []
    
    # Put skills first to ensure they aren't truncated
    skills = [s.get("name", "") for s in candidate.get("skills", [])]
    text_parts.append("Skills: " + ", ".join(skills))
    
    # Add summary
    profile = candidate.get("profile", {})
    text_parts.append(profile.get("summary", ""))
    
    # Add only the most recent 2 roles to save token space
    for role in candidate.get("career_history", [])[:2]:
        text_parts.append(role.get("title", ""))
        text_parts.append(role.get("description", ""))
        
    full_text = " ".join(filter(None, text_parts))
    
    # Truncate to roughly 2,000 characters to stay within the 512 token limit of BART
    return full_text[:2000]

def main():
    texts = []
    true_labels = []
    candidate_ids = []

    print(f"[*] Loading dataset from {GOLDEN_SET_PATH} (merged with {CANDIDATE_POOL_FILES})...")
    records = list(load_golden_set())
    sampled_records = random.sample(records, k=min(100, len(records)))

    for record in sampled_records:
        parsed = record.get("parsed", {})
        tech_fit = parsed.get("Tech_Fit")

        if tech_fit is not None:
            candidate_ids.append(record["candidate_id"])
            texts.append(extract_candidate_text(record))
            true_labels.append(int(tech_fit))

    print(f"[*] Loaded {len(texts)} labeled candidates.")

    # 1. Define the logical hypotheses for the NLI model to evaluate
    # These must read like natural language sentences.
    tier_mapping = {
        "This person works in non-engineering roles like marketing, HR, or operations.": 0,
        "This person primarily builds computer vision, speech, or robotics applications.": 1,
        "This person is a general machine learning, backend, or data engineer.": 2,
        "This person builds semantic search, vector embeddings, or retrieval pipelines.": 3,
        "This person evaluates and owns production ranking, search, or recommendation systems.": 4
    }
    
    candidate_labels = list(tier_mapping.keys())

    # 2. Initialize the Zero-Shot NLI Pipeline
    # Automatically use GPU if available
    device = 0 if torch.cuda.is_available() else -1
    print("\n[*] Loading facebook/bart-large-mnli (this may take a moment)...")
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

    pred_labels = []

    # 3. Run Inference
    print("[*] Running Zero-Shot Classification on Golden Set...")
    for idx, text in enumerate(texts):
        # We use multi_class=False because a candidate can only belong to one primary tier
        result = classifier(text, candidate_labels=candidate_labels)
        
        # The model returns labels sorted by probability. We take the top one.
        top_string_label = result['labels'][0]
        top_prob = result['scores'][0]
        
        predicted_tier = tier_mapping[top_string_label]
        pred_labels.append(predicted_tier)
        
        if idx % 10 == 0 and idx > 0:
            print(f"    Processed {idx}/{len(texts)} candidates...")

    # 4. Evaluate the Model
    print("\n[*] Evaluating NLI Model...")
    accuracy = accuracy_score(true_labels, pred_labels)
    mae = mean_absolute_error(true_labels, pred_labels)
    
    print("=" * 40)
    print(" ── APPROACH 3: ZERO-SHOT NLI METRICS ── ")
    print("=" * 40)
    print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")
    print(f"Mean Absolute Error:  {mae:.4f}")
    print("=" * 40)
    
    print("\nConfusion Matrix (Rows: True Tier, Cols: Predicted Tier):")
    print(confusion_matrix(true_labels, pred_labels))

if __name__ == "__main__":
    main()

[*] Loading dataset from ../data\golden_set.jsonl (merged with ['../data\\train_set.jsonl', '../data\\validation_set.jsonl', '../data\\test_set.jsonl'])...
[*] Loaded 100 labeled candidates.

[*] Loading facebook/bart-large-mnli (this may take a moment)...


Device set to use cpu


[*] Running Zero-Shot Classification on Golden Set...
    Processed 10/100 candidates...
    Processed 20/100 candidates...
    Processed 30/100 candidates...
    Processed 40/100 candidates...
    Processed 50/100 candidates...
    Processed 60/100 candidates...
    Processed 70/100 candidates...
    Processed 80/100 candidates...
    Processed 90/100 candidates...

[*] Evaluating NLI Model...
 ── APPROACH 3: ZERO-SHOT NLI METRICS ── 
Exact Match Accuracy: 65.00%
Mean Absolute Error:  0.6000

Confusion Matrix (Rows: True Tier, Cols: Predicted Tier):
[[ 4  0  0  3  4]
 [ 0  0  2  1  0]
 [ 0  0 61 19  2]
 [ 0  0  0  0  0]
 [ 0  0  4  0  0]]


- The SLM (Small Language Model) Classifier

In [6]:
import os
import json
import re
from llama_cpp import Llama
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix

def extract_candidate_text(candidate):
    """Combines text fields. We can allow much more text here (up to 4k tokens)."""
    text_parts = []
    
    skills = [s.get("name", "") for s in candidate.get("skills", [])]
    text_parts.append("SKILLS:\n" + ", ".join(skills))
    
    profile = candidate.get("profile", {})
    text_parts.append("\nSUMMARY:\n" + profile.get("summary", ""))
    
    text_parts.append("\nCAREER HISTORY:")
    for role in candidate.get("career_history", []):
        text_parts.append(f"- Title: {role.get('title', '')}")
        text_parts.append(f"  Description: {role.get('description', '')}")
        
    # Cap at ~3000 chars just to keep inference lightning fast
    return "\n".join(filter(None, text_parts))[:3000]

def build_messages(candidate_text):
    """Constructs a strict Zero-Shot chat history for Qwen/Llama models.

    Qwen3 is a "thinking" model: by default it emits a <think>...</think>
    reasoning block before its final answer. With a tiny max_tokens budget
    that block gets truncated mid-thought and the digit is never produced.
    We append "/no_think" (Qwen3's documented soft switch) to the user turn
    to ask it to skip reasoning and answer directly.
    """
    return [
        {
            "role": "system",
            "content": (
                "You are an expert AI recruiter.\n"
                "Assign exactly one Tech_Fit score.\n\n"
                "Rubric:\n"
                "4 = Built or owned production retrieval, ranking, recommendation or search systems.\n"
                "3 = Built production RAG, semantic search or embedding systems.\n"
                "2 = General ML, backend or data engineering.\n"
                "1 = Computer Vision, Speech or Robotics.\n"
                "0 = Non-engineering roles.\n\n"
                "Return ONLY one integer from 0 to 4.\n"
                "No explanation."
            ),
        },
        {
            "role": "user",
            "content": candidate_text + "\n\n/no_think",
        },
    ]

def extract_score(raw_output):
    """Parse the predicted Tech_Fit digit out of a model response.

    Even with /no_think, some quantized builds still leak a <think> block --
    so strip it first and search only the text AFTER the closing tag. This
    also avoids false-positive matches against digits mentioned inside the
    rubric text itself if it gets echoed back during reasoning.
    """
    text = raw_output
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    match = re.search(r"\b([0-4])\b", text)
    return int(match.group(1)) if match else None

def main():
    # Adjust path and execution parameters for your hardware
    model_path = "../models/Qwen3-4B-Q4_K_M.gguf"

    # 1. Load Golden Set Data (merged with full candidate records)
    texts = []
    true_labels = []
    candidate_ids = []

    print(f"[*] Loading dataset from {GOLDEN_SET_PATH} (merged with {CANDIDATE_POOL_FILES})...")

    for record in load_golden_set():
        parsed = record.get("parsed", {})
        tech_fit = parsed.get("Tech_Fit")

        # Skip records without a valid Tech_Fit label
        if tech_fit is None:
            continue

        candidate_ids.append(record["candidate_id"])
        texts.append(extract_candidate_text(record))
        true_labels.append(int(tech_fit))

    if not texts:
        print("[!] No valid labels found. Check your JSON keys.")
        return

    # Choose 100 samples deterministically
    sample_size = min(100, len(texts))
    sample_indices = random.Random(42).sample(range(len(texts)), sample_size)

    texts = [texts[i] for i in sample_indices]
    true_labels = [true_labels[i] for i in sample_indices]
    candidate_ids = [candidate_ids[i] for i in sample_indices]

    print(f"[*] Loaded {len(texts)} labeled candidates.")

    # 1. Initialize Local LLM
    print(f"[*] Loading Local SLM from {model_path}...")
    llm = Llama(
        model_path=model_path,
        n_ctx=4096,
        n_gpu_layers=0,              # Set to -1 if using CUDA/GPU
        n_threads=os.cpu_count(),    # Maximize CPU usage
        verbose=False                # Suppress llama.cpp C-level logs
    )

    pred_labels = []

    # 2. Run Inference with Retry Logic
    print("[*] Running Zero-Shot Scoring...")
    for idx, text in enumerate(texts):
        messages = build_messages(text)
        
        pred = None
        raw_output = ""
        
        # Try twice: Initial attempt + 1 correction pass
        for attempt in range(2):
            response = llm.create_chat_completion(
                messages=messages,
                temperature=0.0,
                # Qwen3's <think> block needs real budget to complete even
                # with /no_think requested -- 3 tokens guarantees truncation
                # mid-thought (finish_reason="length") and an unparseable
                # output every single time.
                max_tokens=200,
            )

            raw_output = response["choices"][0]["message"]["content"].strip()
            print("=" * 80)
            print(f"Candidate: {candidate_ids[idx]}")
            print(f"Ground Truth: {true_labels[idx]}")
            print("\nSLM Output:")
            print(raw_output)
            print("=" * 80)

            pred = extract_score(raw_output)

            if pred is not None:
                break
                
            # Second attempt: append the failure and explicitly correct the model
            messages.append(
                {
                    "role": "assistant",
                    "content": raw_output,
                }
            )
            
            messages.append(
                {
                    "role": "user",
                    "content": (
                        "Your previous answer was invalid.\n"
                        "Return ONLY one digit: 0, 1, 2, 3 or 4. /no_think"
                    ),
                }
            )
            
        # Fallback if both attempts fail
        if pred is None:
            print(f"[!] Could not parse response for {candidate_ids[idx]}.")
            print(f"    Final Output: {raw_output}")
            pred = 2
            
        pred_labels.append(pred)
        
        if (idx + 1) % 10 == 0:
            print(f"    Processed {idx + 1}/{len(texts)} candidates...")

    # 3. Evaluate the Model
    print("\n[*] Evaluating Local SLM...")
    accuracy = accuracy_score(true_labels, pred_labels)
    mae = mean_absolute_error(true_labels, pred_labels)
    
    print("=" * 40)
    print(" ── APPROACH 4: LOCAL SLM METRICS (CHAT API) ── ")
    print("=" * 40)
    print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")
    print(f"Mean Absolute Error:  {mae:.4f}")
    print("=" * 40)
    
    print("\nConfusion Matrix (Rows: True Tier, Cols: Predicted Tier):")
    print(confusion_matrix(true_labels, pred_labels))

if __name__ == "__main__":
    main()

[*] Loading dataset from ../data\golden_set.jsonl (merged with ['../data\\train_set.jsonl', '../data\\validation_set.jsonl', '../data\\test_set.jsonl'])...
[*] Loaded 100 labeled candidates.
[*] Loading Local SLM from ../models/Qwen3-4B-Q4_K_M.gguf...
[*] Running Zero-Shot Scoring...
Candidate: CAND_0052765
Ground Truth: 2

SLM Output:
<think>

</think>

2
Candidate: CAND_0083852
Ground Truth: 4

SLM Output:
<think>

</think>

3
Candidate: CAND_0067220
Ground Truth: 2

SLM Output:
<think>

</think>

1
Candidate: CAND_0004534
Ground Truth: 1

SLM Output:
<think>

</think>

2
Candidate: CAND_0009903
Ground Truth: 2

SLM Output:
<think>

</think>

2
Candidate: CAND_0030212
Ground Truth: 0

SLM Output:
<think>

</think>

1
Candidate: CAND_0094445
Ground Truth: 2

SLM Output:
<think>

</think>

3
Candidate: CAND_0024027
Ground Truth: 2

SLM Output:
<think>

</think>

1
Candidate: CAND_0046309
Ground Truth: 2

SLM Output:
<think>

</think>

3
Candidate: CAND_0055189
Ground Truth: 0

SLM Outp

Context Approaches

- SLM + LOGIC TREE

In [7]:
import os
import json
import random
from llama_cpp import Llama
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix

# Assuming INPUT_FILE is defined in your environment, e.g.:
# INPUT_FILE = "golden_set.jsonl" 

SAMPLE_SIZE = 100

def calculate_context_fit(extracted_data):
    """
    Applies the strict Context_Fit rubric using extracted JSON data.
    """
    total_yoe = extracted_data.get("total_yoe_years", 0)
    product_months = extracted_data.get("total_product_company_months", 0)
    longest_tenure = extracted_data.get("longest_tenure_months", 0)
    avg_tenure = extracted_data.get("average_tenure_months", 0)

    is_pure_academic = extracted_data.get("is_pure_academic", False)
    is_pure_consulting = extracted_data.get("is_pure_consulting", False)
    is_management = extracted_data.get("is_management_heavy", False)
    visa_required = extracted_data.get("requires_visa_sponsorship", False)

    if is_pure_academic or is_pure_consulting or (avg_tenure < 18) or visa_required:
        return 0

    if is_management or (longest_tenure < 24):
        return 1

    if (5 <= total_yoe <= 9) and (product_months >= 48) and (longest_tenure >= 36):
        return 4

    if product_months >= 24 and longest_tenure >= 24:
        return 3

    return 2

def extract_candidate_text(candidate):
    text_parts = []

    profile = candidate.get("profile", {})
    text_parts.append(f"Location/Visa: {profile.get('location', '')}, {profile.get('country', '')}")
    text_parts.append(f"Total YOE: {profile.get('years_of_experience', 0)}")

    text_parts.append("\nCAREER HISTORY:")
    for role in candidate.get("career_history", []):
        text_parts.append(f"- Company: {role.get('company', '')} ({role.get('industry', '')})")
        text_parts.append(f"  Title: {role.get('title', '')}")
        text_parts.append(f"  Tenure: {role.get('duration_months', 0)} months")

    return "\n".join(filter(None, text_parts))[:2500]

def build_messages(candidate_text):
    return [
        {
            "role": "system",
            "content": (
                "You are an expert technical recruiter. Extract the requested metrics "
                "from the candidate's career history.\n"
                "- Consulting firms include TCS, Wipro, Infosys, Cognizant, etc.\n"
                "- Product companies build their own software.\n"
                "- Management roles include Director, VP, and Engineering Manager.\n"
                "Output ONLY valid JSON."
            )
        },
        {
            "role": "user",
            "content": f"Candidate Data:\n{candidate_text}"
        }
    ]

def main():
    candidates_path = INPUT_FILE
    model_path = "../models/Qwen3-4B-Q4_K_M.gguf"

    print(f"[*] Loading Local SLM from {model_path}...")
    llm = Llama(
        model_path=model_path,
        n_ctx=4096,
        n_gpu_layers=0,
        n_threads=os.cpu_count(),
        verbose=False
    )

    json_schema = {
        "type": "object",
        "properties": {
            "total_yoe_years": {"type": "number"},
            "total_product_company_months": {"type": "integer"},
            "longest_tenure_months": {"type": "integer"},
            "average_tenure_months": {"type": "integer"},
            "is_pure_academic": {"type": "boolean"},
            "is_pure_consulting": {"type": "boolean"},
            "is_management_heavy": {"type": "boolean"},
            "requires_visa_sponsorship": {"type": "boolean"}
        },
        "required": [
            "total_yoe_years", "total_product_company_months",
            "longest_tenure_months", "average_tenure_months",
            "is_pure_academic", "is_pure_consulting",
            "is_management_heavy", "requires_visa_sponsorship"
        ]
    }

    print("[*] Running JSON Extraction on candidates...")

    true_labels = []
    pred_labels = []

    try:
        # --- ADDED: Load all valid records into memory first ---
        valid_records = []
        for candidate in load_golden_set():
            parsed = candidate.get("parsed", {})
            true_context = parsed.get("Context_Fit")

            if true_context is not None:
                valid_records.append(candidate)

        if not valid_records:
            print("[!] No valid labels found. Check your JSON keys.")
            return

        # --- ADDED: Choose exactly SAMPLE_SIZE candidates deterministically ---
        sample_size = min(SAMPLE_SIZE, len(valid_records))
        sampled_records = random.Random(42).sample(valid_records, sample_size)

        print(f"[*] Loaded {len(valid_records)} valid candidates. Sampled {sample_size} for evaluation.")

        processed = 0
        for candidate in sampled_records:
            cid = candidate["candidate_id"]
            true_context = candidate.get("parsed", {}).get("Context_Fit")

            text = extract_candidate_text(candidate)
            messages = build_messages(text)

            extracted_data = None
            raw_output = ""
            context_score = 2  # Default fallback score

            for _ in range(2):
                response = llm.create_chat_completion(
                    messages=messages,
                    temperature=0.0,
                    max_tokens=250,
                    response_format={
                        "type": "json_object",
                        "schema": json_schema
                    }
                )

                raw_output = response["choices"][0]["message"]["content"].strip()

                try:
                    extracted_data = json.loads(raw_output)
                    break
                except json.JSONDecodeError:
                    messages.append({"role": "assistant", "content": raw_output})
                    messages.append({
                        "role": "user",
                        "content": "Your previous output was not valid JSON. Please return ONLY valid JSON matching the exact schema."
                    })

            if extracted_data is None:
                print(f"[!] Failed to parse JSON for {cid} after 2 attempts.")
                print(f"    Final Output: {raw_output}")
                print(f"[{cid}] Calculated Context_Fit: {context_score} (Fallback)\n")
            else:
                context_score = calculate_context_fit(extracted_data)
                print(f"[{cid}] Extracted JSON: {json.dumps(extracted_data)}")
                print(f"[{cid}] Calculated Context_Fit: {context_score}\n")

            true_labels.append(int(true_context))
            pred_labels.append(context_score)

            processed += 1
            if processed % 10 == 0:
                print(f"    Processed {processed}/{sample_size} candidates...")

        if true_labels and pred_labels:
            print("\n[*] Evaluating SLM Context_Fit Model...")
            accuracy = accuracy_score(true_labels, pred_labels)
            mae = mean_absolute_error(true_labels, pred_labels)
            
            print("=" * 45)
            print(" ── APPROACH 3: SLM + LOGIC TREE METRICS ── ")
            print("=" * 45)
            print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")
            print(f"Mean Absolute Error:  {mae:.4f}")
            print("=" * 45)
            
            print("\nConfusion Matrix (Rows: True Tier, Cols: Predicted Tier):")
            print(confusion_matrix(true_labels, pred_labels))
        else:
            print("\n[!] No valid labels found to calculate metrics.")

    except FileNotFoundError:
        print(f"[!] Error: Could not find {candidates_path}.")

if __name__ == "__main__":
    main()

[*] Loading Local SLM from ../models/Qwen3-4B-Q4_K_M.gguf...
[*] Running JSON Extraction on candidates...
[*] Loaded 800 valid candidates. Sampled 100 for evaluation.
[CAND_0052765] Extracted JSON: {"total_yoe_years": 9.8, "total_product_company_months": 0, "longest_tenure_months": 36, "average_tenure_months": 25, "is_pure_academic": false, "is_pure_consulting": false, "is_management_heavy": false, "requires_visa_sponsorship": true}
[CAND_0052765] Calculated Context_Fit: 0

[CAND_0083852] Extracted JSON: {"total_yoe_years": 6.0, "total_product_company_months": 32, "longest_tenure_months": 39, "average_tenure_months": 35, "is_pure_academic": false, "is_pure_consulting": false, "is_management_heavy": false, "requires_visa_sponsorship": false}
[CAND_0083852] Calculated Context_Fit: 3

[CAND_0067220] Extracted JSON: {"total_yoe_years": 5.1, "total_product_company_months": 0, "longest_tenure_months": 25, "average_tenure_months": 20, "is_pure_academic": false, "is_pure_consulting": false, "i

Final Ranking approaches

In [5]:
import json
import numpy as np
import pandas as pd
from datetime import date
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import xgboost as xgb

# ── 1. Shared helpers ────────────────────────────────────────────────────────

AS_OF_DATE_STR = "2026-05-27"   # max(last_active_date) across the pool

_CONSULTING = {
    "tcs", "infosys", "wipro", "cognizant", "capgemini", "accenture",
    "mphasis", "hexaware", "tech mahindra", "l&t infotech", "hcl technologies"
}
_CONSULTING_INDUSTRIES = {"consulting", "it services", "it services and consulting"}
_MGMT_TITLES = ["director", "vp ", "vice president", "engineering manager", "head of", "cto", "ceo"]
SIX_MONTHS_DAYS = 180
OPEN_TO_WORK_CAP = 3


# ── 2. Tech_Fit: extract text for embedding (Approach 2 text format) ────────

def tech_extract_text(candidate):
    profile = candidate.get("profile", {})
    parts = [profile.get("headline", ""), profile.get("summary", "")]
    for role in candidate.get("career_history", []):
        parts.append(role.get("title", ""))
        parts.append(role.get("description", ""))
    for skill in candidate.get("skills", []):
        parts.append(skill.get("name", ""))
    return " ".join(filter(None, parts))


# ── 3. Context_Fit: deterministic extractor + logic tree (Approach 3) ───────

def compute_context_fit_deterministic(candidate):
    """Apply the Context_Fit rubric directly from structured fields,
    matching the exact logic tree in calculate_context_fit() (cell 21)."""
    profile  = candidate.get("profile", {})
    roles    = candidate.get("career_history", [])
    sig      = candidate.get("redrob_signals", {})

    total_yoe = float(profile.get("years_of_experience", 0) or 0)
    durations = [r.get("duration_months", 0) for r in roles]
    avg_tenure     = sum(durations) / len(durations) if durations else 0
    longest_tenure = max(durations) if durations else 0

    def _is_consulting(r):
        c = r.get("company", "").strip().lower()
        ind = r.get("industry", "").strip().lower()
        return c in _CONSULTING or ind in _CONSULTING_INDUSTRIES

    is_pure_consulting = bool(roles) and all(_is_consulting(r) for r in roles)
    is_management = any(
        any(t in r.get("title", "").lower() for t in _MGMT_TITLES)
        for r in roles
    )
    product_months = sum(
        r.get("duration_months", 0) for r in roles if not _is_consulting(r)
    )

    # Visa / relocation logic: only hard-reject if explicitly outside India
    # AND not willing to relocate (mirrors the golden pipeline rubric)
    country     = profile.get("country", "India").strip().lower()
    relocate    = sig.get("willing_to_relocate", True)
    visa_reject = (country != "india") and (not relocate)

    # Hard rejects → 0
    if is_pure_consulting or (avg_tenure < 18) or visa_reject:
        return 0
    # Management-heavy or short tenures → 1
    if is_management or (longest_tenure < 24):
        return 1
    # Ideal target → 4
    if (5 <= total_yoe <= 9) and (product_months >= 48) and (longest_tenure >= 36):
        return 4
    # Strong fit → 3
    if product_months >= 24 and longest_tenure >= 24:
        return 3
    # Standard profile → 2
    return 2


# ── 4. Behavior_Fit: deterministic from redrob_signals ──────────────────────

def compute_behavior_fit(candidate):
    sig     = candidate.get("redrob_signals", {})
    la      = sig.get("last_active_date")
    rr      = sig.get("recruiter_response_rate")
    notice  = sig.get("notice_period_days", 0)
    open_tw = sig.get("open_to_work_flag", False)

    as_of = date.fromisoformat(AS_OF_DATE_STR)
    d = (as_of - date.fromisoformat(la)).days if la else SIX_MONTHS_DAYS
    r = rr if rr is not None else 0.0

    recency  = 4 if d<=14 else 3 if d<=45 else 2 if d<=90 else 1 if d<SIX_MONTHS_DAYS else 0
    response = 4 if r>=0.70 else 3 if r>=0.50 else 2 if r>=0.30 else 1 if r>=0.10 else 0

    base = min(recency, response)
    if not open_tw:
        base = min(base, OPEN_TO_WORK_CAP)
    if notice > 90:
        base = min(base, 2)
    elif notice > 30:
        base = min(base, 3)
    return base


# ── 5. Load golden set and build raw feature matrix ─────────────────────────

print("[*] Loading golden set...")
records = list(load_golden_set())
print(f"    {len(records)} merged records loaded.")

# Text for embedding (Tech_Fit model)
texts      = [tech_extract_text(r) for r in records]
# Ground-truth labels from the golden set
gt_tech    = np.array([int(r["parsed"].get("Tech_Fit", 0))         for r in records])
gt_context = np.array([int(r["parsed"].get("Context_Fit", 0))      for r in records])
gt_behavior= np.array([int(r["parsed"].get("Behavior_Fit", 0))     for r in records])
gt_final   = np.array([int(r["parsed"].get("Final_Score_NDCG", 0)) for r in records])
gt_strength= np.array([int(r["parsed"].get("Profile_Strength_Score", 50)) for r in records])
cids       = [r["candidate_id"] for r in records]


# ── 6. Train Tech_Fit predictor (Approach 2: XGBoost on embeddings) ─────────

print("[*] Generating embeddings (all-MiniLM-L6-v2)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
X_emb = embed_model.encode(texts, show_progress_bar=True, batch_size=64)

# 80/20 split (same seed as Approach 2 cell for reproducibility)
idx_tr, idx_te = train_test_split(
    np.arange(len(records)), test_size=0.2, random_state=42, stratify=gt_tech
)

clf = xgb.XGBClassifier(
    objective="multi:softmax", num_class=5, eval_metric="mlogloss",
    max_depth=3, learning_rate=0.1, n_estimators=100, random_state=42,
    verbosity=0
)
print("[*] Training XGBoost Tech_Fit classifier...")
clf.fit(X_emb[idx_tr], gt_tech[idx_tr])

# Predict over the FULL golden set (train + test) so we have scores for all
# 800 candidates for ranking.  The train-set predictions are in-sample but
# that's intentional: we want a complete ranking surface, not a split one.
pred_tech = clf.predict(X_emb)
print(f"    Tech_Fit accuracy on held-out test (20%): "
      f"{(pred_tech[idx_te] == gt_tech[idx_te]).mean()*100:.1f}%")


# ── 7. Compute Context_Fit and Behavior_Fit deterministically ────────────────

print("[*] Computing Context_Fit (deterministic logic tree)...")
pred_context  = np.array([compute_context_fit_deterministic(r) for r in records])
print(f"    Context_Fit accuracy vs golden labels: "
      f"{(pred_context == gt_context).mean()*100:.1f}%")

print("[*] Computing Behavior_Fit (deterministic from redrob_signals)...")
pred_behavior = np.array([compute_behavior_fit(r) for r in records])
print(f"    Behavior_Fit accuracy vs golden labels: "
      f"{(pred_behavior == gt_behavior).mean()*100:.1f}%")


# ── 8. Assemble candidates_data for the ranking cells ────────────────────────

candidates_data = [
    {
        "candidate_id":        cid,
        "Tech_Fit":            int(tf),
        "Context_Fit":         int(cf),
        "Behavior_Fit":        int(bf),
        "Profile_Strength_Score": int(ps),
        "Final_Score_NDCG":    int(fs),   # ground-truth relevance label
    }
    for cid, tf, cf, bf, ps, fs in zip(
        cids, pred_tech, pred_context, pred_behavior, gt_strength, gt_final
    )
]

# Test-split subset (used for the LambdaMART held-out evaluation)
test_data = [candidates_data[i] for i in idx_te]

print(f"\n[*] candidates_data ready: {len(candidates_data)} total, "
      f"{len(test_data)} held-out test records.")
print("    Distribution of predicted Tech_Fit:   ",
      dict(zip(*np.unique(pred_tech,    return_counts=True))))
print("    Distribution of predicted Context_Fit:",
      dict(zip(*np.unique(pred_context, return_counts=True))))


[*] Loading golden set...
    800 merged records loaded.
[*] Generating embeddings (all-MiniLM-L6-v2)...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

[*] Training XGBoost Tech_Fit classifier...
    Tech_Fit accuracy on held-out test (20%): 98.1%
[*] Computing Context_Fit (deterministic logic tree)...
    Context_Fit accuracy vs golden labels: 54.4%
[*] Computing Behavior_Fit (deterministic from redrob_signals)...
    Behavior_Fit accuracy vs golden labels: 100.0%

[*] candidates_data ready: 800 total, 160 held-out test records.
    Distribution of predicted Tech_Fit:    {np.int32(0): np.int64(172), np.int32(1): np.int64(16), np.int32(2): np.int64(576), np.int32(3): np.int64(3), np.int32(4): np.int64(33)}
    Distribution of predicted Context_Fit: {np.int64(0): np.int64(2), np.int64(1): np.int64(19), np.int64(2): np.int64(77), np.int64(3): np.int64(553), np.int64(4): np.int64(149)}


- Deterministic Multi-Key Lexicographical Sort

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score, average_precision_score
from scipy.stats import spearmanr


def rank_lexicographical(candidates_data):
    """Ranks candidates using a strict multi-key sort.
    Prioritizes Tech_Fit over Context_Fit over Behavior_Fit."""
    df = pd.DataFrame(candidates_data)

    # Candidates with Tech=0 or Context=0 are always ranked last
    df["is_vetoed"] = (df["Tech_Fit"] == 0) | (df["Context_Fit"] == 0)

    df_ranked = df.sort_values(
        by=["is_vetoed", "Tech_Fit", "Context_Fit", "Behavior_Fit"],
        ascending=[True, False, False, False]
    )

    return df_ranked.drop(columns=["is_vetoed"]).reset_index(drop=True)


def evaluate_ranking(ranked_df, approach_name):
    """Compute ranking metrics."""

    ranked_df = ranked_df.copy()
    ranked_df["rank"] = range(1, len(ranked_df) + 1)

    true_relevance = ranked_df["Final_Score_NDCG"].values
    pred_scores = (len(ranked_df) + 1 - ranked_df["rank"]).values

    ndcg_10 = ndcg_score([true_relevance], [pred_scores.astype(float)], k=10)
    ndcg_50 = ndcg_score([true_relevance], [pred_scores.astype(float)], k=50)
    ndcg_all = ndcg_score([true_relevance], [pred_scores.astype(float)])

    rho, _ = spearmanr(true_relevance, pred_scores)

    relevant = (true_relevance >= 3)

    p10 = relevant[:10].mean()
    p20 = relevant[:20].mean()

    map_score = average_precision_score(
        relevant.astype(int),
        pred_scores
    )

    final_score = (
        0.50 * ndcg_10 +
        0.30 * ndcg_50 +
        0.15 * map_score +
        0.05 * p10
    )

    print("=" * 52)
    print(f" ── {approach_name} ──")
    print("=" * 52)
    print(f"  NDCG@10    : {ndcg_10:.4f}")
    print(f"  NDCG@50    : {ndcg_50:.4f}")
    print(f"  NDCG@all   : {ndcg_all:.4f}")
    print(f"  MAP        : {map_score:.4f}")
    print(f"  Spearman ρ : {rho:.4f}")
    print(f"  P@10       : {p10:.4f} ({int(relevant[:10].sum())}/10)")
    print(f"  P@20       : {p20:.4f} ({int(relevant[:20].sum())}/20)")
    print("-" * 52)
    print(f"  Final Score: {final_score:.4f}")
    print("=" * 52)

    print("\nTop-10 ranked candidates:")
    print(f"{'Rank':<6}{'Candidate ID':<16}{'Tech':>5}{'Ctx':>5}{'Beh':>5}{'GT':>5}")
    print("-" * 42)

    for _, row in ranked_df.head(10).iterrows():
        print(
            f"{int(row['rank']):<6}"
            f"{row['candidate_id']:<16}"
            f"{int(row['Tech_Fit']):>5}"
            f"{int(row['Context_Fit']):>5}"
            f"{int(row['Behavior_Fit']):>5}"
            f"{int(row['Final_Score_NDCG']):>5}"
        )

    return {
        "NDCG@10": ndcg_10,
        "NDCG@50": ndcg_50,
        "NDCG@all": ndcg_all,
        "MAP": map_score,
        "Spearman": rho,
        "P@10": p10,
        "P@20": p20,
        "Final Score": final_score,
    }


# ── Run on real predictions ────────────────────────────────────────────────
print("[*] Running Lexicographical Sort on real predictions...\n")

ranked_lex = rank_lexicographical(candidates_data)

lex_metrics = evaluate_ranking(
    ranked_lex,
    "APPROACH A: LEXICOGRAPHICAL SORT METRICS"
)

[*] Running Lexicographical Sort on real predictions...

 ── APPROACH A: LEXICOGRAPHICAL SORT METRICS ──
  NDCG@10    : 0.9826
  NDCG@50    : 0.9776
  NDCG@all   : 0.9967
  MAP        : 0.8874
  Spearman ρ : 0.7715
  P@10       : 1.0000 (10/10)
  P@20       : 0.9000 (18/20)
----------------------------------------------------
  Final Score: 0.9677

Top-10 ranked candidates:
Rank  Candidate ID     Tech  Ctx  Beh   GT
------------------------------------------
1     CAND_0068932        4    4    4    4
2     CAND_0002025        4    4    4    4
3     CAND_0083852        4    4    3    4
4     CAND_0082973        4    4    3    4
5     CAND_0062772        4    4    3    4
6     CAND_0096142        4    4    2    4
7     CAND_0099269        4    4    2    4
8     CAND_0073917        4    4    2    3
9     CAND_0087630        4    4    2    4
10    CAND_0099751        4    4    2    4


- Weighted Nonlinear Scoring Function

In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score, average_precision_score
from scipy.stats import spearmanr


def rank_nonlinear_weighted(candidates_data, wt=0.5, wc=0.3, wb=0.2):
    """Ranks candidates using a vectorized mathematical formula.
    Rank Score = delta * (wt*Tech + wc*Context + wb*Behavior + Profile_Strength/100)"""
    df = pd.DataFrame(candidates_data)
    delta = np.where((df["Tech_Fit"] == 0) | (df["Context_Fit"] == 0), 0, 1)
    df["rank_score"] = delta * (
        (wt * df["Tech_Fit"]) +
        (wc * df["Context_Fit"]) +
        (wb * df["Behavior_Fit"])
    )
    return df.sort_values(by="rank_score", ascending=False).reset_index(drop=True)


# ── Run on real predictions ──────────────────────────────────────────────────
print("[*] Running Weighted Nonlinear Scoring on real predictions...\n")
ranked_wt = rank_nonlinear_weighted(candidates_data)

# Reuse evaluate_ranking from the Lex cell (already in namespace)
ranked_wt["rank"] = range(1, len(ranked_wt) + 1)
true_relevance = ranked_wt["Final_Score_NDCG"].values
pred_scores    = (len(ranked_wt) + 1 - ranked_wt["rank"]).values

ndcg_10  = ndcg_score([true_relevance], [pred_scores.astype(float)], k=10)
ndcg_50  = ndcg_score([true_relevance], [pred_scores.astype(float)], k=50)
ndcg_all = ndcg_score([true_relevance], [pred_scores.astype(float)])

rho, _ = spearmanr(true_relevance, pred_scores)

# Binary relevance (>=3 considered relevant)
relevant = (true_relevance >= 3)

# Precision@K
p10 = relevant[:10].mean()
p20 = relevant[:20].mean()

# Mean Average Precision (MAP)
map_score = average_precision_score(relevant.astype(int), pred_scores)

# Composite ranking score
final_score = (
    0.50 * ndcg_10 +
    0.30 * ndcg_50 +
    0.15 * map_score +
    0.05 * p10
)

print("=" * 52)
print(" ── APPROACH B: WEIGHTED NONLINEAR SCORING METRICS ──")
print("=" * 52)
print(f"  NDCG@10    : {ndcg_10:.4f}")
print(f"  NDCG@50    : {ndcg_50:.4f}")
print(f"  NDCG@all   : {ndcg_all:.4f}")
print(f"  MAP        : {map_score:.4f}")
print(f"  Spearman ρ : {rho:.4f}")
print(f"  P@10       : {p10:.4f}  ({int(relevant[:10].sum())}/10 relevant in top-10)")
print(f"  P@20       : {p20:.4f}  ({int(relevant[:20].sum())}/20 relevant in top-20)")
print("-" * 52)
print(f"  Final Score: {final_score:.4f}")
print("=" * 52)

wt_metrics = {
    "NDCG@10": ndcg_10,
    "NDCG@50": ndcg_50,
    "NDCG@all": ndcg_all,
    "MAP": map_score,
    "Spearman": rho,
    "P@10": p10,
    "P@20": p20,
    "Final Score": final_score,
}

# Save final ranking to CSV
ranked_wt.to_csv("../data/final_ranking.csv", index=False)
print("[*] Saved final ranking to final_ranking.csv")

[*] Running Weighted Nonlinear Scoring on real predictions...

 ── APPROACH B: WEIGHTED NONLINEAR SCORING METRICS ──
  NDCG@10    : 1.0000
  NDCG@50    : 0.9797
  NDCG@all   : 0.9970
  MAP        : 0.9070
  Spearman ρ : 0.7667
  P@10       : 1.0000  (10/10 relevant in top-10)
  P@20       : 0.9000  (18/20 relevant in top-20)
----------------------------------------------------
  Final Score: 0.9799
[*] Saved final ranking to final_ranking.csv


- Local Learning-to-Rank (LTR) with XGBoost

In [8]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score, average_precision_score
from scipy.stats import spearmanr


def extract_features(record):
    """Extract all numerical signals into a flat dictionary."""
    parsed = record.get("parsed", {})
    profile = record.get("profile", {})
    redrob = record.get("redrob_signals", {})

    target_ndcg = parsed.get("Final_Score_NDCG", 0)

    features = {
    # Predicted scores
    "Tech_Fit": record.get("Tech_Fit", parsed.get("Tech_Fit", 0)),
    "Context_Fit": record.get("Context_Fit", parsed.get("Context_Fit", 0)),
    "Behavior_Fit": record.get("Behavior_Fit", parsed.get("Behavior_Fit", 0)),

    # Raw metadata signals
    "Years_of_Experience": float(profile.get("years_of_experience", 0) or 0),
    "Notice_Period": redrob.get("notice_period_days", 90),
    "GitHub_Score": max(redrob.get("github_activity_score", 0), 0),
    "Response_Rate": redrob.get("recruiter_response_rate", 0) or 0,
    "Open_to_Work": int(redrob.get("open_to_work_flag", False)),
    }

    return features, int(target_ndcg), record["candidate_id"]


def main():
    print(f"[*] Building feature matrix from {len(candidates_data)} candidates...")

    X_list, y_list, cid_list = [], [], []

    for record in candidates_data:
        full_record = next(
            (r for r in records if r["candidate_id"] == record["candidate_id"]),
            {},
        )

        # Predicted scores override original values
        merged = {**full_record, **record}

        features, target, cid = extract_features(merged)

        X_list.append(features)
        y_list.append(target)
        cid_list.append(cid)

    df_X = pd.DataFrame(X_list)
    y = np.array(y_list)

    # Train / Test split
    X_train, X_test, y_train, y_test, cid_train, cid_test = train_test_split(
        df_X,
        y,
        cid_list,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    # All candidates belong to the same query (single JD)
    qid_train = np.ones(len(X_train), dtype=np.int32)
    qid_test = np.ones(len(X_test), dtype=np.int32)

    print(f"\n[*] Training XGBRanker (LambdaMART) on {len(X_train)} samples...")

    ranker = xgb.XGBRanker(
        tree_method="hist",
        objective="rank:ndcg",
        lambdarank_num_pair_per_sample=8,
        max_depth=4,
        learning_rate=0.1,
        n_estimators=150,
        random_state=42,
    )

    ranker.fit(
        X_train,
        y_train,
        qid=qid_train,
        eval_set=[(X_test, y_test)],
        eval_qid=[qid_test],
        verbose=20,
    )

    print("\n[*] Running Inference on Test Set...")

    y_pred = ranker.predict(X_test)

    # Ranking metrics
    ndcg_10 = ndcg_score([y_test], [y_pred], k=10)
    ndcg_50 = ndcg_score([y_test], [y_pred], k=50)
    ndcg_all = ndcg_score([y_test], [y_pred])

    rho, _ = spearmanr(y_test, y_pred)

    # Binary relevance
    relevant = (y_test >= 3)

    # Sort by predicted ranking
    order = np.argsort(y_pred)[::-1]
    rel_sorted = relevant[order]

    # Precision
    p10 = rel_sorted[:10].mean()
    p20 = rel_sorted[:20].mean()

    # MAP
    map_score = average_precision_score(
        relevant.astype(int),
        y_pred
    )

    # Composite Final Score
    final_score = (
        0.50 * ndcg_10 +
        0.30 * ndcg_50 +
        0.15 * map_score +
        0.05 * p10
    )

    print("=" * 60)
    print(" ── APPROACH C: LambdaMART (LTR) METRICS ──")
    print("=" * 60)
    print(f"  NDCG@10      : {ndcg_10:.4f}")
    print(f"  NDCG@50      : {ndcg_50:.4f}")
    print(f"  NDCG@all     : {ndcg_all:.4f}")
    print(f"  MAP          : {map_score:.4f}")
    print(f"  Spearman ρ   : {rho:.4f}")
    print(f"  P@10         : {p10:.4f} ({int(rel_sorted[:10].sum())}/10)")
    print(f"  P@20         : {p20:.4f} ({int(rel_sorted[:20].sum())}/20)")
    print("-" * 60)
    print(f"  Final Score  : {final_score:.4f}")
    print("=" * 60)

    print("\n[*] Feature Importances (Weight):")
    for fname, weight in sorted(
        zip(df_X.columns, ranker.feature_importances_),
        key=lambda x: -x[1],
    ):
        if weight > 0:
            print(f"  {fname:<25}: {weight:.4f}")

    model_save_path = "../models/xgboost_ranker.json"
    ranker.save_model(model_save_path)
    print(f"\n[*] Model saved to {model_save_path}")

    ltr_metrics = {
        "NDCG@10": ndcg_10,
        "NDCG@50": ndcg_50,
        "NDCG@all": ndcg_all,
        "MAP": map_score,
        "Spearman": rho,
        "P@10": p10,
        "P@20": p20,
        "Final Score": final_score,
    }

if __name__ == "__main__":
    main()


[*] Building feature matrix from 800 candidates...

[*] Training XGBRanker (LambdaMART) on 640 samples...
[0]	validation_0-ndcg@8:0.26026
[20]	validation_0-ndcg@8:0.96873
[40]	validation_0-ndcg@8:0.96873
[60]	validation_0-ndcg@8:0.96873
[80]	validation_0-ndcg@8:0.88715
[100]	validation_0-ndcg@8:0.88715
[120]	validation_0-ndcg@8:0.88715
[140]	validation_0-ndcg@8:0.88715
[149]	validation_0-ndcg@8:0.88715

[*] Running Inference on Test Set...
 ── APPROACH C: LambdaMART (LTR) METRICS ──
  NDCG@10      : 0.9192
  NDCG@50      : 0.8921
  NDCG@all     : 0.9715
  MAP          : 0.6828
  Spearman ρ   : 0.6181
  P@10         : 0.5000 (5/10)
  P@20         : 0.2500 (5/20)
------------------------------------------------------------
  Final Score  : 0.8546

[*] Feature Importances (Weight):
  Tech_Fit                 : 0.5660
  Response_Rate            : 0.2569
  Behavior_Fit             : 0.0896
  Open_to_Work             : 0.0671
  GitHub_Score             : 0.0204

[*] Model saved to ../models/

Reasoning Approaches

- Extractive Vector Summarization

In [19]:
import os
import json
import re
import csv
import torch
from sentence_transformers import SentenceTransformer, util

# ── Paths ────────────────────────────────────────────────────────────────────
# Input:  best-strategy CSV produced by final_ranking_test.py
RANKED_CSV_PATH   = "../data/final_ranking.csv"
# Candidate pool to look up full profiles (carrier_history lives here)
POOL_PATH         = "../data/candidates.jsonl"
JD_TXT_PATH       = "../data/job_description.txt"
JD_DOCX_PATH      = "../data/job_description.docx"
# Output: same CSV enriched with extractive reasoning
OUTPUT_CSV_PATH   = "../data/final_ranking_with_reasoning.csv"


def load_job_description():
    if os.path.exists(JD_TXT_PATH):
        with open(JD_TXT_PATH, "r", encoding="utf-8") as f:
            return f.read().strip()
    if os.path.exists(JD_DOCX_PATH):
        import docx
        doc = docx.Document(JD_DOCX_PATH)
        return "\n".join(p.text for p in doc.paragraphs).strip()
    raise FileNotFoundError(f"No JD found at {JD_TXT_PATH} or {JD_DOCX_PATH}")


def load_candidate_pool(path):
    """Load full candidate profiles keyed by candidate_id."""
    pool = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                pool[rec["candidate_id"]] = rec
    return pool


def get_best_sentence(candidate, jd_embedding, model):
    """
    Extracts all sentences from career history, embeds them, and
    returns the one with the highest cosine similarity to the JD.
    """
    sentences = []
    for role in candidate.get("career_history", []):
        desc = role.get("description", "")
        if desc:
            sentences.extend(re.split(r'(?<=[.!?]) +', desc))

    valid = [s.strip() for s in sentences if len(s.split()) > 5]

    if not valid:
        return "Candidate has foundational experience but limited descriptive text."

    sent_embs   = model.encode(valid, convert_to_tensor=True)
    cosine_scores = util.cos_sim(jd_embedding, sent_embs)[0]
    best_idx    = torch.argmax(cosine_scores).item()
    return valid[best_idx]


def generate_extractive_reasoning():
    print("[*] Loading SentenceTransformer...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    print("[*] Loading Job Description...")
    jd_text    = load_job_description()
    jd_embedding = model.encode(jd_text, convert_to_tensor=True)

    print(f"[*] Loading candidate pool from {POOL_PATH}...")
    pool = load_candidate_pool(POOL_PATH)

    print(f"[*] Reading ranked candidates from {RANKED_CSV_PATH}...")
    rows = []
    with open(RANKED_CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i >= 100:
                break
            rows.append(row)
    print(f"[*] Generating extractive reasoning for {len(rows)} candidates...\n")
    enriched = []
    for row in rows:
        cid       = row["candidate_id"]
        candidate = pool.get(cid, {})

        best_sentence = get_best_sentence(candidate, jd_embedding, model)

        # Build a recruiter-friendly one-liner
        profile = candidate.get("profile", {})
        sig     = candidate.get("redrob_signals", {})
        yoe     = profile.get("years_of_experience", "")
        loc     = profile.get("location", profile.get("country", ""))
        notice  = sig.get("notice_period_days")

        parts = []
        if yoe:    parts.append(f"{yoe}y exp")
        if loc:    parts.append(f"{loc}-based")
        if notice and isinstance(notice, int) and notice > 60:
            parts.append(f"notice {notice}d")
        meta = "; ".join(parts)

        reasoning = best_sentence + (f" [{meta}]" if meta else "")
        print(f"[{cid}] {reasoning}\n")
        row["reasoning"] = reasoning
        enriched.append(row)

    # Save enriched CSV
    with open(OUTPUT_CSV_PATH, "w", newline="", encoding="utf-8") as f:
        fieldnames = list(enriched[0].keys()) if enriched else []
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(enriched)

    print(f"[*] Saved enriched CSV → {OUTPUT_CSV_PATH}")


if __name__ == "__main__":
    generate_extractive_reasoning()

generate_extractive_reasoning()


[*] Loading SentenceTransformer...
[*] Loading Job Description...
[*] Loading candidate pool from ../data/candidates.jsonl...
[*] Reading ranked candidates from ../data/final_ranking.csv...
[*] Generating extractive reasoning for 100 candidates...

[CAND_0068932] Built recommendation-style features at a mid-stage startup — lighter weight than ranking systems at FAANG, but production. [5.2y exp; Noida, Uttar Pradesh-based]

[CAND_0002025] Built and shipped a production recommendation system at a marketplace product, going from offline experimentation to live A/B test in 5 months. [5.9y exp; Trivandrum, Kerala-based]

[CAND_0082973] Built recommendation-style features at a mid-stage startup — lighter weight than ranking systems at FAANG, but production. [6.4y exp; Chandigarh, Chandigarh-based]

[CAND_0062772] My main role was engineering: building the Flask-based prediction API, integrating with the feature store, and writing the model-serving observability layer. [6.6y exp; Trivandrum, 

- Local SLM (Small Language Model)

In [21]:
import json
import csv
import os
from llama_cpp import Llama

# ── Paths ────────────────────────────────────────────────────────────────────
RANKED_CSV_PATH = "../data/final_ranking.csv"
POOL_PATH       = "../data/candidates.jsonl"
MODEL_PATH      = "../models/Qwen3-4B-Q4_K_M.gguf"
OUTPUT_CSV_PATH = "../data/final_ranking_slm_reasoning.csv"


# =============================================================================
# Data helpers
# =============================================================================

def load_candidate_pool(path):
    pool = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                pool[rec["candidate_id"]] = rec
    return pool


def extract_candidate_summary(candidate):
    """Hyper-condensed candidate view for the SLM (≤1 k tokens).

    Includes location, notice period, and company type so the model can
    surface caveats (long notice, non-India, consulting background) in the
    one-sentence reason.
    """
    parts   = []
    profile = candidate.get("profile", {})
    sig     = candidate.get("redrob_signals", {})

    yoe     = profile.get("years_of_experience", 0)
    loc     = profile.get("location", profile.get("country", "Unknown"))
    notice  = sig.get("notice_period_days", "unknown")
    otw     = sig.get("open_to_work_flag", False)

    parts.append(f"Experience: {yoe} years.")
    parts.append(f"Location: {loc}.")
    parts.append(f"Notice period: {notice} days.")
    parts.append(f"Open to work: {'yes' if otw else 'no'}.")

    skills = [s.get("name", "") for s in candidate.get("skills", [])][:6]
    parts.append(f"Top Skills: {', '.join(skills)}.")

    career = candidate.get("career_history", [])
    for i, r in enumerate(career[:2]):      # most recent two roles
        company_type = r.get("industry", "")
        parts.append(
            f"Role {i+1}: {r.get('title')} at {r.get('company')} "
            f"({company_type}, {r.get('duration_months', 0)} months). "
            f"Details: {r.get('description', '')[:300]}"
        )
    return " ".join(parts)[:1200]


# =============================================================================
# Prompt construction
# =============================================================================

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM = (
    "You are an expert technical recruiter writing a one-sentence summary "
    "for a candidate ranked for a Senior AI Engineer role (Python, Vector DBs, "
    "Production Retrieval/Ranking). "
    "Rules:\n"
    "1. Write EXACTLY ONE sentence, 25–45 words.\n"
    "2. State the strongest technical evidence first (specific tool, project, "
    "metric, or achievement).\n"
    "3. If there is a meaningful caveat — long notice period (>90 days), "
    "consulting-only background, below-band YOE, or no direct retrieval work — "
    "name it honestly in the same sentence.\n"
    "4. End with bracketed metadata: [Xy exp; City-based; notice Nd] "
    "omitting any item you do not know.\n"
    "5. Do NOT invent information. Do NOT use bullet points, greetings, or prefixes."
)

# ── Five calibrated few-shot examples ────────────────────────────────────────
# The five examples deliberately span the realistic output distribution:
#   Ex 1 — Tier 4 candidate: strong retrieval ownership + metric → strong positive
#   Ex 2 — Tier 3 candidate: good ML at product company, no quality measurement
#   Ex 3 — Tier 2 with notice caveat: solid engineering but long notice
#   Ex 4 — Consulting background: recent product pivot but weak fit history
#   Ex 5 — Bottom-of-list filler: adjacent skills only, low YOE
#
# Each example:
#   • uses a different caveat type so the model learns to flag different issues
#   • ends with the [metadata] tag used throughout the pipeline
#   • stays within the 25–45 word constraint
#   • names a specific tool / metric / role detail, not just a category

FEW_SHOTS = [

    # ── Ex 1: Tier 4 — Owns production retrieval with measured quality ─────────
    {
        "user": (
            "Candidate Data:\n"
            "Experience: 7 years. Location: Pune, Maharashtra. "
            "Notice period: 30 days. Open to work: yes. "
            "Top Skills: Python, FAISS, LangChain, XGBoost, FastAPI, Elasticsearch.\n"
            "Role 1: Senior ML Engineer at TechRetail (E-commerce, 41 months). "
            "Details: Owned end-to-end semantic search and product-ranking pipeline "
            "serving 2.5M daily queries; ran monthly offline NDCG evaluations and "
            "A/B experiments that lifted CTR by 14%.\n\n"
            "/no_think"
        ),
        "assistant": (
            "Owned a 2.5M-query/day semantic search and ranking pipeline at a "
            "product e-commerce company, with monthly NDCG evaluations and A/B "
            "experiments lifting CTR by 14%; strong retrieval match. "
            "[7y exp; Pune-based; notice 30d]"
        ),
    },

    # ── Ex 2: Tier 3 — Strong ML at product co, no explicit quality measurement ─
    {
        "user": (
            "Candidate Data:\n"
            "Experience: 6 years. Location: Bengaluru, Karnataka. "
            "Notice period: 60 days. Open to work: yes. "
            "Top Skills: Python, PyTorch, Recommendation Systems, Redis, Kubernetes.\n"
            "Role 1: ML Engineer at MusicStream (SaaS, 36 months). "
            "Details: Built collaborative-filtering and embedding-based recommendation "
            "models powering the 'Discover Weekly' feature for 800 k active users; "
            "deployed via FastAPI on GCP.\n\n"
            "/no_think"
        ),
        "assistant": (
            "Built embedding-based recommendation models for 800 k users at a "
            "product SaaS company over three years; strong retrieval-adjacent work, "
            "though no evidence of offline ranking-quality evaluation. "
            "[6y exp; Bengaluru-based; notice 60d]"
        ),
    },

    # ── Ex 3: Tier 2 — General ML / backend, long notice period ─────────────────
    {
        "user": (
            "Candidate Data:\n"
            "Experience: 9 years. Location: Hyderabad, Telangana. "
            "Notice period: 120 days. Open to work: no. "
            "Top Skills: Python, Spark, Airflow, PostgreSQL, Docker.\n"
            "Role 1: Senior Data Engineer at FinCo (Fintech, 48 months). "
            "Details: Designed and maintained ETL pipelines for risk-scoring models; "
            "migrated batch jobs to Spark, reducing processing time by 40%.\n\n"
            "/no_think"
        ),
        "assistant": (
            "Nine years as a senior data engineer with strong Python and Spark "
            "skills, but no evidence of retrieval, ranking, or recommendation work; "
            "long notice (120 days) and not actively looking add placement risk. "
            "[9y exp; Hyderabad-based; notice 120d]"
        ),
    },

    # ── Ex 4: Consulting-heavy background, recent product pivot ──────────────────
    {
        "user": (
            "Candidate Data:\n"
            "Experience: 5 years. Location: Chennai, Tamil Nadu. "
            "Notice period: 45 days. Open to work: yes. "
            "Top Skills: Python, TensorFlow, SQL, Java, Azure.\n"
            "Role 1: AI Engineer at ProductCo (SaaS, 12 months). "
            "Details: Fine-tuned BERT models for document classification; "
            "integrated outputs into an internal search UI.\n"
            "Role 2: Software Engineer at Infosys (IT Services, 48 months). "
            "Details: Developed enterprise data migration scripts and REST APIs "
            "for banking clients.\n\n"
            "/no_think"
        ),
        "assistant": (
            "Recent pivot to AI at a product company (12 months) with BERT "
            "fine-tuning and search integration, but four prior years at Infosys "
            "in IT services weakens the product-company and retrieval-depth fit. "
            "[5y exp; Chennai-based; notice 45d]"
        ),
    },

    # ── Ex 5: Bottom-of-list filler — adjacent skills only, low YOE ─────────────
    {
        "user": (
            "Candidate Data:\n"
            "Experience: 3 years. Location: Delhi, India. "
            "Notice period: 30 days. Open to work: yes. "
            "Top Skills: Python, Django, React, MySQL, REST APIs.\n"
            "Role 1: Backend Engineer at StartupX (SaaS, 36 months). "
            "Details: Built and maintained REST APIs for a B2B invoicing platform; "
            "wrote unit tests and managed CI/CD pipelines on AWS.\n\n"
            "/no_think"
        ),
        "assistant": (
            "Three years as a backend engineer with solid Python and Django skills "
            "but no AI, ML, retrieval, or ranking experience; included as tail "
            "filler given strong recency signals and quick availability. "
            "[3y exp; Delhi-based; notice 30d]"
        ),
    },

]


def build_messages(candidate_summary):
    """Build the full few-shot chat history + actual candidate turn."""
    messages = [{"role": "system", "content": SYSTEM}]

    for ex in FEW_SHOTS:
        messages.append({"role": "user",      "content": ex["user"]})
        messages.append({"role": "assistant", "content": ex["assistant"]})

    # Actual candidate
    messages.append({
        "role": "user",
        "content": f"Candidate Data:\n{candidate_summary}\n\n/no_think",
    })
    return messages


# =============================================================================
# Main inference loop
# =============================================================================

def generate_slm_reasoning():
    print(f"[*] Loading Qwen3-4B from {MODEL_PATH}...")
    llm = Llama(
        model_path=MODEL_PATH,
        n_ctx=4096,              # larger context to fit the five few-shot turns
        n_gpu_layers=0,          # ← set to -1 to use GPU if available
        n_threads=os.cpu_count(),
        verbose=False,
    )

    print(f"[*] Loading candidate pool from {POOL_PATH}...")
    pool = load_candidate_pool(POOL_PATH)

    print(f"[*] Reading ranked candidates from {RANKED_CSV_PATH}...")
    rows = []
    with open(RANKED_CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i >= 10:
                break
            rows.append(row)

    print(f"[*] Generating SLM reasoning for {len(rows)} candidates...\n")
    enriched = []
    for row in rows:
        cid       = row["candidate_id"]
        candidate = pool.get(cid, {})
        summary   = extract_candidate_summary(candidate)
        messages  = build_messages(summary)

        response = llm.create_chat_completion(
            messages=messages,
            temperature=0.2,     # slightly lower for more consistent output
            max_tokens=120,      # ~45 words + metadata tag with headroom
        )

        raw = response["choices"][0]["message"]["content"].strip()

        # Strip any residual <think>…</think> block just in case
        if "</think>" in raw:
            raw = raw.split("</think>", 1)[1].strip()

        # Remove surrounding quotes the model may add
        reasoning = raw.strip('"').strip("'").strip()
        print(f"[{cid}] {reasoning}\n")
        row["reasoning"] = reasoning
        enriched.append(row)

    # Save enriched CSV
    with open(OUTPUT_CSV_PATH, "w", newline="", encoding="utf-8") as f:
        fieldnames = list(enriched[0].keys()) if enriched else []
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(enriched)

    print(f"[*] Saved SLM reasoning CSV → {OUTPUT_CSV_PATH}")


if __name__ == "__main__":
    generate_slm_reasoning()

generate_slm_reasoning()

[*] Loading Qwen3-4B from ../models/Qwen3-4B-Q4_K_M.gguf...
[*] Loading candidate pool from ../data/candidates.jsonl...
[*] Reading ranked candidates from ../data/final_ranking.csv...
[*] Generating SLM reasoning for 10 candidates...

[CAND_0068932] Built production recommendation features using collaborative filtering and gradient-boosted re-ranking at a mid-stage AI startup; used Milvus for vector storage but no direct retrieval/ranking evaluation metrics; 5.2y exp; Noida-based; notice 30d.

[CAND_0002025] Built a production recommendation system at Apple marketplace in 5 months, transitioning from offline experiments to live A/B tests; also fine-tuned LLaMA-2 and Mistral for candidate-JD matching with 200K high-quality pairs; strong FAISS, TensorFlow, and OpenSearch expertise. [5.9y exp; Trivandrum-based; notice 30d]

[CAND_0082973] Built computer vision models for image moderation using PyTorch and ResNet variants on 200K images, with production inference service; also built lightw